# Análise Descritiva e Comparativa - Processo Seletivo de Bolsas

## Comparação de Perfis de Alunos Selecionados vs Não Selecionados

**Autor:** [Seu Nome]  
**Número USP:** [Seu número USP]  
**Data:** Abril de 2026

---

### Objetivo da Análise

Este relatório apresenta uma análise descritiva e inferencial dos dados de 3.000 alunos que participaram de um processo seletivo para bolsas. O objetivo principal é comparar o perfil dos candidatos selecionados com o dos candidatos não selecionados, considerando as variáveis demográficas disponíveis: gênero, orientação sexual, idade, raça/cor e deficiência.

As questões centrais que buscamos responder são:
- Existem diferenças significativas no perfil demográfico entre alunos selecionados e não selecionados?
- Há padrões de representação que sugiram possíveis vieses no processo de seleção?
- Quais variáveis mais influenciam a probabilidade de seleção?

## 1. Carregamento e Exploração do Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Carregar o dataset
df = pd.read_csv('ProcessoSeletivo.csv', encoding='utf-8')

# Exibir informações básicas sobre o dataset
print("=" * 80)
print("INFORMAÇÕES BÁSICAS DO DATASET")
print("=" * 80)
print(f"\nDimensões do dataset: {df.shape[0]} linhas × {df.shape[1]} colunas\n")
print("Tipos de dados:")
print(df.dtypes)
print("\n" + "=" * 80)
print("Primeiras 10 linhas do dataset:")
print("=" * 80)
df.head(10)

In [ ]:
# Verificar dados faltantes
print("\n" + "=" * 80)
print("VERIFICAÇÃO DE DADOS FALTANTES")
print("=" * 80)
missing_data = pd.DataFrame({
    'Coluna': df.columns,
    'Dados Faltantes': df.isnull().sum().values,
    'Percentual (%)': (df.isnull().sum().values / len(df) * 100).round(2)
})
print(missing_data)

# Estatísticas descritivas gerais
print("\n" + "=" * 80)
print("ESTATÍSTICAS DESCRITIVAS - VARIÁVEIS NUMÉRICAS")
print("=" * 80)
print(df[['Idade']].describe().round(2))

# Distribuição da variável alvo
print("\n" + "=" * 80)
print("DISTRIBUIÇÃO DE SELECIONADOS")
print("=" * 80)
dist_selecao = df['Selecionado'].value_counts()
print(dist_selecao)
print(f"\nPercentual de Selecionados: {(dist_selecao['Sim'] / len(df) * 100):.2f}%")
print(f"Percentual de Não Selecionados: {(dist_selecao['Não'] / len(df) * 100):.2f}%")

## 2. Estatísticas Descritivas por Status de Seleção

In [ ]:
# Estatísticas por grupo de seleção
print("=" * 80)
print("ESTATÍSTICAS DESCRITIVAS - IDADE POR STATUS DE SELEÇÃO")
print("=" * 80)

stats_idade = df.groupby('Selecionado')['Idade'].describe().round(2)
print(stats_idade)

# Criar tabela comparativa
print("\n" + "=" * 80)
print("RESUMO COMPARATIVO - IDADE")
print("=" * 80)
comparison_idade = pd.DataFrame({
    'Grupo': ['Selecionados', 'Não Selecionados'],
    'N': [df[df['Selecionado'] == 'Sim'].shape[0], df[df['Selecionado'] == 'Não'].shape[0]],
    'Idade Média': [
        df[df['Selecionado'] == 'Sim']['Idade'].mean().round(2),
        df[df['Selecionado'] == 'Não']['Idade'].mean().round(2)
    ],
    'Idade Mediana': [
        df[df['Selecionado'] == 'Sim']['Idade'].median(),
        df[df['Selecionado'] == 'Não']['Idade'].median()
    ],
    'Desvio Padrão': [
        df[df['Selecionado'] == 'Sim']['Idade'].std().round(2),
        df[df['Selecionado'] == 'Não']['Idade'].std().round(2)
    ],
    'Mínima': [
        df[df['Selecionado'] == 'Sim']['Idade'].min(),
        df[df['Selecionado'] == 'Não']['Idade'].min()
    ],
    'Máxima': [
        df[df['Selecionado'] == 'Sim']['Idade'].max(),
        df[df['Selecionado'] == 'Não']['Idade'].max()
    ]
})
print(comparison_idade.to_string(index=False))

In [ ]:
# Análise de variáveis categóricas
print("\n" + "=" * 80)
print("DISTRIBUIÇÃO POR SEXO - POR STATUS DE SELEÇÃO")
print("=" * 80)

sexo_crosstab = pd.crosstab(df['Sexo'], df['Selecionado'], margins=True)
sexo_pct = pd.crosstab(df['Sexo'], df['Selecionado'], normalize='index') * 100
print("\nContagem Absoluta:")
print(sexo_crosstab)
print("\nPercentual por Sexo (em %):")
print(sexo_pct.round(2))

print("\n" + "=" * 80)
print("DISTRIBUIÇÃO POR ORIENTAÇÃO SEXUAL - POR STATUS DE SELEÇÃO")
print("=" * 80)

ori_crosstab = pd.crosstab(df['OrientacaoSexual'], df['Selecionado'], margins=True)
ori_pct = pd.crosstab(df['OrientacaoSexual'], df['Selecionado'], normalize='index') * 100
print("\nContagem Absoluta:")
print(ori_crosstab)
print("\nPercentual por Orientação Sexual (em %):")
print(ori_pct.round(2))

In [ ]:
print("\n" + "=" * 80)
print("DISTRIBUIÇÃO POR RAÇA/COR - POR STATUS DE SELEÇÃO")
print("=" * 80)

raca_crosstab = pd.crosstab(df['Raça-cor'], df['Selecionado'], margins=True)
raca_pct = pd.crosstab(df['Raça-cor'], df['Selecionado'], normalize='index') * 100
print("\nContagem Absoluta:")
print(raca_crosstab)
print("\nPercentual por Raça/Cor (em %):")
print(raca_pct.round(2))

print("\n" + "=" * 80)
print("DISTRIBUIÇÃO DE DEFICIÊNCIA - POR STATUS DE SELEÇÃO")
print("=" * 80)

# Categorizar deficiência em "Possui" ou "Não Possui"
df['Tem_Deficiencia'] = df['Deficiência'].apply(lambda x: 'Não' if x == 'Não possuo' else 'Sim')

deficiencia_crosstab = pd.crosstab(df['Tem_Deficiencia'], df['Selecionado'], margins=True)
deficiencia_pct = pd.crosstab(df['Tem_Deficiencia'], df['Selecionado'], normalize='index') * 100
print("\nContagem Absoluta:")
print(deficiencia_crosstab)
print("\nPercentual por Deficiência (em %):")
print(deficiencia_pct.round(2))

## 3. Comparação de Perfis Demográficos

In [1]:
# Criar tabela comparativa resumida
print("=" * 100)
print("PERFIL DEMOGRÁFICO RESUMIDO - SELECIONADOS vs NÃO SELECIONADOS")
print("=" * 100)

selecionados = df[df['Selecionado'] == 'Sim']
nao_selecionados = df[df['Selecionado'] == 'Não']

perfil_comparativo = pd.DataFrame({
    'Demográfico': [
        'Total de Alunos',
        'Idade Média (anos)',
        'Idade Mediana (anos)',
        '% Mulheres (F)',
        '% Homens (M)',
        '% Heterossexual',
        '% Bissexual',
        '% Homossexual',
        '% Pansexual',
        '% Prefiro não responder',
        '% Branca',
        '% Parda',
        '% Preta',
        '% Amarela',
        '% Indígena',
        '% Não informada',
        '% Com Deficiência',
        '% Sem Deficiência'
    ],
    'Selecionados': [
        len(selecionados),
        f"{selecionados['Idade'].mean():.2f}",
        f"{selecionados['Idade'].median():.0f}",
        f"{(selecionados['Sexo'] == 'F').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['Sexo'] == 'M').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['OrientacaoSexual'] == 'Heterossexual').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['OrientacaoSexual'] == 'Bissexual').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['OrientacaoSexual'] == 'Homossexual').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['OrientacaoSexual'] == 'Pansexual').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['OrientacaoSexual'] == 'Prefiro não responder').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['Raça-cor'] == 'Branca').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['Raça-cor'] == 'Parda').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['Raça-cor'] == 'Preta').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['Raça-cor'] == 'Amarela').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['Raça-cor'] == 'Indígena').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['Raça-cor'] == 'Não informada').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['Tem_Deficiencia'] == 'Sim').sum() / len(selecionados) * 100:.1f}",
        f"{(selecionados['Tem_Deficiencia'] == 'Não').sum() / len(selecionados) * 100:.1f}"
    ],
    'Não Selecionados': [
        len(nao_selecionados),
        f"{nao_selecionados['Idade'].mean():.2f}",
        f"{nao_selecionados['Idade'].median():.0f}",
        f"{(nao_selecionados['Sexo'] == 'F').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['Sexo'] == 'M').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['OrientacaoSexual'] == 'Heterossexual').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['OrientacaoSexual'] == 'Bissexual').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['OrientacaoSexual'] == 'Homossexual').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['OrientacaoSexual'] == 'Pansexual').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['OrientacaoSexual'] == 'Prefiro não responder').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['Raça-cor'] == 'Branca').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['Raça-cor'] == 'Parda').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['Raça-cor'] == 'Preta').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['Raça-cor'] == 'Amarela').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['Raça-cor'] == 'Indígena').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['Raça-cor'] == 'Não informada').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['Tem_Deficiencia'] == 'Sim').sum() / len(nao_selecionados) * 100:.1f}",
        f"{(nao_selecionados['Tem_Deficiencia'] == 'Não').sum() / len(nao_selecionados) * 100:.1f}"
    ]
})

print(perfil_comparativo.to_string(index=False))

PERFIL DEMOGRÁFICO RESUMIDO - SELECIONADOS vs NÃO SELECIONADOS


NameError: name 'df' is not defined

## 4. Análise de Distribuições de Variáveis-Chave

In [ ]:
# Visualizações - Distribuição de Idade
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de idade por grupo
df[df['Selecionado'] == 'Sim']['Idade'].hist(bins=25, alpha=0.6, label='Selecionados', ax=axes[0], color='green', edgecolor='black')
df[df['Selecionado'] == 'Não']['Idade'].hist(bins=25, alpha=0.6, label='Não Selecionados', ax=axes[0], color='red', edgecolor='black')
axes[0].set_xlabel('Idade (anos)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequência', fontsize=11, fontweight='bold')
axes[0].set_title('Distribuição de Idade por Status de Seleção', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot de idade
df.boxplot(column='Idade', by='Selecionado', ax=axes[1])
axes[1].set_xlabel('Status de Seleção', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Idade (anos)', fontsize=11, fontweight='bold')
axes[1].set_title('Box Plot de Idade por Status de Seleção', fontsize=12, fontweight='bold')
axes[1].get_figure().suptitle('')

plt.tight_layout()
plt.show()

print("Os gráficos acima mostram a distribuição etária de alunos selecionados e não selecionados.")

In [ ]:
# Visualizações - Sexo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sexo_counts = pd.crosstab(df['Sexo'], df['Selecionado'])
sexo_counts.plot(kind='bar', ax=axes[0], color=['lightcoral', 'lightgreen'], edgecolor='black', width=0.7)
axes[0].set_xlabel('Sexo', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Número de Alunos', fontsize=11, fontweight='bold')
axes[0].set_title('Distribuição por Sexo', fontsize=12, fontweight='bold')
axes[0].legend(['Não Selecionados', 'Selecionados'], loc='upper right')
axes[0].set_xticklabels(['Feminino', 'Masculino'], rotation=0)
axes[0].grid(True, alpha=0.3, axis='y')

# Percentual por sexo
sexo_pct = pd.crosstab(df['Sexo'], df['Selecionado'], normalize='index') * 100
sexo_pct.plot(kind='bar', ax=axes[1], color=['lightcoral', 'lightgreen'], edgecolor='black', width=0.7)
axes[1].set_xlabel('Sexo', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Percentual (%)', fontsize=11, fontweight='bold')
axes[1].set_title('Taxa de Seleção por Sexo', fontsize=12, fontweight='bold')
axes[1].legend(['Não Selecionados', 'Selecionados'], loc='upper right')
axes[1].set_xticklabels(['Feminino', 'Masculino'], rotation=0)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Visualizações - Raça/Cor
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

raca_counts = pd.crosstab(df['Raça-cor'], df['Selecionado'])
raca_counts.plot(kind='bar', ax=axes[0], color=['lightcoral', 'lightgreen'], edgecolor='black')
axes[0].set_xlabel('Raça/Cor', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Número de Alunos', fontsize=11, fontweight='bold')
axes[0].set_title('Distribuição por Raça/Cor', fontsize=12, fontweight='bold')
axes[0].legend(['Não Selecionados', 'Selecionados'], loc='upper right')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Taxa de seleção por raça
raca_pct = pd.crosstab(df['Raça-cor'], df['Selecionado'], normalize='index') * 100
raca_pct_sel = raca_pct['Sim'].sort_values(ascending=False)
axes[1].barh(range(len(raca_pct_sel)), raca_pct_sel.values, color='lightgreen', edgecolor='black')
axes[1].set_yticks(range(len(raca_pct_sel)))
axes[1].set_yticklabels(raca_pct_sel.index)
axes[1].set_xlabel('Taxa de Seleção (%)', fontsize=11, fontweight='bold')
axes[1].set_title('Taxa de Seleção por Raça/Cor', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')
axes[1].axvline(x=50, color='gray', linestyle='--', alpha=0.5, label='50%')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualizações - Orientação Sexual
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ori_counts = pd.crosstab(df['OrientacaoSexual'], df['Selecionado'])
ori_counts.plot(kind='bar', ax=axes[0], color=['lightcoral', 'lightgreen'], edgecolor='black')
axes[0].set_xlabel('Orientação Sexual', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Número de Alunos', fontsize=11, fontweight='bold')
axes[0].set_title('Distribuição por Orientação Sexual', fontsize=12, fontweight='bold')
axes[0].legend(['Não Selecionados', 'Selecionados'], loc='upper right')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Taxa de seleção por orientação
ori_pct = pd.crosstab(df['OrientacaoSexual'], df['Selecionado'], normalize='index') * 100
ori_pct_sel = ori_pct['Sim'].sort_values(ascending=False)
axes[1].barh(range(len(ori_pct_sel)), ori_pct_sel.values, color='lightgreen', edgecolor='black')
axes[1].set_yticks(range(len(ori_pct_sel)))
axes[1].set_yticklabels(ori_pct_sel.index)
axes[1].set_xlabel('Taxa de Seleção (%)', fontsize=11, fontweight='bold')
axes[1].set_title('Taxa de Seleção por Orientação Sexual', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')
axes[1].axvline(x=50, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Visualizações - Deficiência
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

def_counts = pd.crosstab(df['Tem_Deficiencia'], df['Selecionado'])
def_counts.plot(kind='bar', ax=axes[0], color=['lightcoral', 'lightgreen'], edgecolor='black', width=0.7)
axes[0].set_xlabel('Tem Deficiência', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Número de Alunos', fontsize=11, fontweight='bold')
axes[0].set_title('Distribuição por Status de Deficiência', fontsize=12, fontweight='bold')
axes[0].legend(['Não Selecionados', 'Selecionados'], loc='upper right')
axes[0].set_xticklabels(['Não', 'Sim'], rotation=0)
axes[0].grid(True, alpha=0.3, axis='y')

# Percentual
def_pct = pd.crosstab(df['Tem_Deficiencia'], df['Selecionado'], normalize='index') * 100
def_pct.plot(kind='bar', ax=axes[1], color=['lightcoral', 'lightgreen'], edgecolor='black', width=0.7)
axes[1].set_xlabel('Tem Deficiência', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Percentual (%)', fontsize=11, fontweight='bold')
axes[1].set_title('Taxa de Seleção por Status de Deficiência', fontsize=12, fontweight='bold')
axes[1].legend(['Não Selecionados', 'Selecionados'], loc='upper right')
axes[1].set_xticklabels(['Não', 'Sim'], rotation=0)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 5. Testes Estatísticos para Diferenças entre Grupos

Nesta seção, realizamos testes estatísticos inferenciais para determinar se as diferenças observadas entre os grupos de selecionados e não-selecionados são estatisticamente significativas.

In [ ]:
# Teste t para a idade
selecionados_idade = df[df['Selecionado'] == 'Sim']['Idade']
nao_selecionados_idade = df[df['Selecionado'] == 'Não']['Idade']

t_stat_idade, p_value_idade = ttest_ind(selecionados_idade, nao_selecionados_idade)

print("=" * 100)
print("TESTE T - DIFERENÇA DE IDADE ENTRE GRUPOS")
print("=" * 100)
print(f"Hipótese Nula: A idade média dos selecionados é igual à dos não-selecionados")
print(f"Estatística T: {t_stat_idade:.4f}")
print(f"P-value: {p_value_idade:.6f}")
print(f"Significância: {'Significante (α=0.05)' if p_value_idade < 0.05 else 'Não significante (α=0.05)'}")
if p_value_idade < 0.05:
    print(f"Conclusão: Há diferença estatisticamente significante na idade entre grupos (p < 0.05)")
else:
    print(f"Conclusão: Não há diferença estatisticamente significante na idade entre grupos (p ≥ 0.05)")

# Teste Chi-quadrado para Sexo
sexo_contingency = pd.crosstab(df['Sexo'], df['Selecionado'])
chi2_sexo, p_sexo, dof_sexo, expected_sexo = chi2_contingency(sexo_contingency)

print("\n" + "=" * 100)
print("TESTE CHI-QUADRADO - SEXO")
print("=" * 100)
print(f"Hipótese Nula: Sexo e seleção são independentes")
print(f"Chi-quadrado: {chi2_sexo:.4f}")
print(f"P-value: {p_sexo:.6f}")
print(f"Significância: {'Significante (α=0.05)' if p_sexo < 0.05 else 'Não significante (α=0.05)'}")
if p_sexo < 0.05:
    print(f"Conclusão: Há associação significante entre sexo e seleção (p < 0.05)")
else:
    print(f"Conclusão: Não há associação significante entre sexo e seleção (p ≥ 0.05)")

In [ ]:
# Teste Chi-quadrado para Orientação Sexual
ori_contingency = pd.crosstab(df['OrientacaoSexual'], df['Selecionado'])
chi2_ori, p_ori, dof_ori, expected_ori = chi2_contingency(ori_contingency)

print("\n" + "=" * 100)
print("TESTE CHI-QUADRADO - ORIENTAÇÃO SEXUAL")
print("=" * 100)
print(f"Hipótese Nula: Orientação sexual e seleção são independentes")
print(f"Chi-quadrado: {chi2_ori:.4f}")
print(f"P-value: {p_ori:.6f}")
print(f"Significância: {'Significante (α=0.05)' if p_ori < 0.05 else 'Não significante (α=0.05)'}")
if p_ori < 0.05:
    print(f"Conclusão: Há associação significante entre orientação sexual e seleção (p < 0.05)")
else:
    print(f"Conclusão: Não há associação significante entre orientação sexual e seleção (p ≥ 0.05)")

# Teste Chi-quadrado para Raça/Cor
raca_contingency = pd.crosstab(df['Raça-cor'], df['Selecionado'])
chi2_raca, p_raca, dof_raca, expected_raca = chi2_contingency(raca_contingency)

print("\n" + "=" * 100)
print("TESTE CHI-QUADRADO - RAÇA/COR")
print("=" * 100)
print(f"Hipótese Nula: Raça/cor e seleção são independentes")
print(f"Chi-quadrado: {chi2_raca:.4f}")
print(f"P-value: {p_raca:.6f}")
print(f"Significância: {'Significante (α=0.05)' if p_raca < 0.05 else 'Não significante (α=0.05)'}")
if p_raca < 0.05:
    print(f"Conclusão: Há associação significante entre raça/cor e seleção (p < 0.05)")
    print(f"Este resultado sugere que pode haver disparidades na seleção relacionadas à raça/cor")
else:
    print(f"Conclusão: Não há associação significante entre raça/cor e seleção (p ≥ 0.05)")

In [ ]:
# Teste Chi-quadrado para Deficiência
def_contingency = pd.crosstab(df['Tem_Deficiencia'], df['Selecionado'])
chi2_def, p_def, dof_def, expected_def = chi2_contingency(def_contingency)

print("\n" + "=" * 100)
print("TESTE CHI-QUADRADO - DEFICIÊNCIA")
print("=" * 100)
print(f"Hipótese Nula: Deficiência e seleção são independentes")
print(f"Chi-quadrado: {chi2_def:.4f}")
print(f"P-value: {p_def:.6f}")
print(f"Significância: {'Significante (α=0.05)' if p_def < 0.05 else 'Não significante (α=0.05)'}")
if p_def < 0.05:
    print(f"Conclusão: Há associação significante entre deficiência e seleção (p < 0.05)")
else:
    print(f"Conclusão: Não há associação significante entre deficiência e seleção (p ≥ 0.05)")

# Resumo dos testes
print("\n" + "=" * 100)
print("RESUMO DOS TESTES ESTATÍSTICOS")
print("=" * 100)
resumo_testes = pd.DataFrame({
    'Variável': ['Idade (Teste T)', 'Sexo (Chi-quadrado)', 'Orientação Sexual (Chi-quadrado)', 'Raça/Cor (Chi-quadrado)', 'Deficiência (Chi-quadrado)'],
    'Estatística': [f"{t_stat_idade:.4f}", f"{chi2_sexo:.4f}", f"{chi2_ori:.4f}", f"{chi2_raca:.4f}", f"{chi2_def:.4f}"],
    'P-value': [f"{p_value_idade:.6f}", f"{p_sexo:.6f}", f"{p_ori:.6f}", f"{p_raca:.6f}", f"{p_def:.6f}"],
    'Significante (α=0.05)': [
        'Sim' if p_value_idade < 0.05 else 'Não',
        'Sim' if p_sexo < 0.05 else 'Não',
        'Sim' if p_ori < 0.05 else 'Não',
        'Sim' if p_raca < 0.05 else 'Não',
        'Sim' if p_def < 0.05 else 'Não'
    ]
})
print(resumo_testes.to_string(index=False))

## 6. Matriz de Comparação Visual

In [ ]:
# Matriz de comparação percentual por característica
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Matriz de Comparação: Taxa de Seleção por Características Demográficas', 
             fontsize=14, fontweight='bold', y=0.995)

# Sexo - Taxa
ax1 = axes[0, 0]
sexo_taxa = pd.crosstab(df['Sexo'], df['Selecionado'], normalize='index') * 100
sexo_taxa_sel = sexo_taxa['Sim']
colors_sexo = ['#2ecc71' if x >= 50 else '#e74c3c' for x in sexo_taxa_sel.values]
ax1.bar(['Feminino', 'Masculino'], sexo_taxa_sel.values, color=colors_sexo, edgecolor='black', width=0.6)
ax1.axhline(y=50, color='gray', linestyle='--', alpha=0.5, linewidth=2)
ax1.set_ylabel('Taxa de Seleção (%)', fontsize=11, fontweight='bold')
ax1.set_title('Sexo', fontsize=12, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(sexo_taxa_sel.values):
    ax1.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

# Raça/Cor - Taxa
ax2 = axes[0, 1]
raca_taxa = pd.crosstab(df['Raça-cor'], df['Selecionado'], normalize='index') * 100
raca_taxa_sel = raca_taxa['Sim'].sort_values(ascending=False)
colors_raca = ['#2ecc71' if x >= 50 else '#e74c3c' for x in raca_taxa_sel.values]
ax2.barh(range(len(raca_taxa_sel)), raca_taxa_sel.values, color=colors_raca, edgecolor='black')
ax2.axvline(x=50, color='gray', linestyle='--', alpha=0.5, linewidth=2)
ax2.set_yticks(range(len(raca_taxa_sel)))
ax2.set_yticklabels(raca_taxa_sel.index, fontsize=10)
ax2.set_xlabel('Taxa de Seleção (%)', fontsize=11, fontweight='bold')
ax2.set_title('Raça/Cor', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 100)
ax2.grid(True, alpha=0.3, axis='x')
for i, v in enumerate(raca_taxa_sel.values):
    ax2.text(v + 1, i, f'{v:.1f}%', va='center', fontweight='bold')

# Orientação Sexual - Taxa
ax3 = axes[1, 0]
ori_taxa = pd.crosstab(df['OrientacaoSexual'], df['Selecionado'], normalize='index') * 100
ori_taxa_sel = ori_taxa['Sim'].sort_values(ascending=False)
colors_ori = ['#2ecc71' if x >= 50 else '#e74c3c' for x in ori_taxa_sel.values]
ax3.barh(range(len(ori_taxa_sel)), ori_taxa_sel.values, color=colors_ori, edgecolor='black')
ax3.axvline(x=50, color='gray', linestyle='--', alpha=0.5, linewidth=2)
ax3.set_yticks(range(len(ori_taxa_sel)))
ax3.set_yticklabels(ori_taxa_sel.index, fontsize=10)
ax3.set_xlabel('Taxa de Seleção (%)', fontsize=11, fontweight='bold')
ax3.set_title('Orientação Sexual', fontsize=12, fontweight='bold')
ax3.set_xlim(0, 100)
ax3.grid(True, alpha=0.3, axis='x')
for i, v in enumerate(ori_taxa_sel.values):
    ax3.text(v + 1, i, f'{v:.1f}%', va='center', fontweight='bold')

# Deficiência - Taxa
ax4 = axes[1, 1]
def_taxa = pd.crosstab(df['Tem_Deficiencia'], df['Selecionado'], normalize='index') * 100
def_taxa_sel = def_taxa['Sim']
labels_def = ['Sem Deficiência', 'Com Deficiência']
colors_def = ['#2ecc71' if x >= 50 else '#e74c3c' for x in def_taxa_sel.values]
ax4.bar(range(len(labels_def)), def_taxa_sel.values, color=colors_def, edgecolor='black', width=0.6)
ax4.axhline(y=50, color='gray', linestyle='--', alpha=0.5, linewidth=2)
ax4.set_xticks(range(len(labels_def)))
ax4.set_xticklabels(labels_def, fontsize=10)
ax4.set_ylabel('Taxa de Seleção (%)', fontsize=11, fontweight='bold')
ax4.set_title('Status de Deficiência', fontsize=12, fontweight='bold')
ax4.set_ylim(0, 100)
ax4.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(def_taxa_sel.values):
    ax4.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Interpretação e Conclusões

### 7.1 Resumo das Descobertas

Com base na análise dos dados de 3.000 candidatos ao processo seletivo, identificamos os seguintes padrões nos perfis demográficos:

In [ ]:
# Gerar insights
selecionados = df[df['Selecionado'] == 'Sim']
nao_selecionados = df[df['Selecionado'] == 'Não']

taxa_geral = len(selecionados) / len(df) * 100

print("=" * 100)
print("PRINCIPAIS ACHADOS E INTERPRETAÇÕES")
print("=" * 100)

print(f"\n1. TAXA GERAL DE SELEÇÃO:")
print(f"   - Taxa geral: {taxa_geral:.1f}% dos candidatos foram selecionados")
print(f"   - Total de selecionados: {len(selecionados)} alunos")
print(f"   - Total de não selecionados: {len(nao_selecionados)} alunos")

# Diferença de idade
idade_diff = abs(selecionados['Idade'].mean() - nao_selecionados['Idade'].mean())
print(f"\n2. DIFERENÇAS DE IDADE:")
print(f"   - Idade média dos selecionados: {selecionados['Idade'].mean():.2f} anos")
print(f"   - Idade média dos não-selecionados: {nao_selecionados['Idade'].mean():.2f} anos")
print(f"   - Diferença: {idade_diff:.2f} anos")
print(f"   - Resultado do teste T: p-value = {p_value_idade:.6f}")
if p_value_idade < 0.05:
    print(f"   - Conclusão: Diferença SIGNIFICANTE - idade é um fator que diferencia os grupos")
else:
    print(f"   - Conclusão: Diferença NÃO SIGNIFICANTE - idade não diferencia os grupos")

# Análise de gênero
taxa_f_sel = (selecionados['Sexo'] == 'F').sum() / (df['Sexo'] == 'F').sum() * 100
taxa_m_sel = (selecionados['Sexo'] == 'M').sum() / (df['Sexo'] == 'M').sum() * 100
print(f"\n3. DIFERENÇAS POR SEXO:")
print(f"   - Taxa de seleção de Mulheres: {taxa_f_sel:.1f}%")
print(f"   - Taxa de seleção de Homens: {taxa_m_sel:.1f}%")
print(f"   - Diferença: {abs(taxa_f_sel - taxa_m_sel):.1f} pontos percentuais")
print(f"   - Resultado do Chi-quadrado: p-value = {p_sexo:.6f}")
if p_sexo < 0.05:
    print(f"   - Conclusão: Associação SIGNIFICANTE - sexo está associado à seleção")
else:
    print(f"   - Conclusão: Associação NÃO SIGNIFICANTE - sexo não está associado à seleção")

# Análise de raça/cor - identifica maior variação
raca_taxa = pd.crosstab(df['Raça-cor'], df['Selecionado'], normalize='index') * 100
print(f"\n4. DIFERENÇAS POR RAÇA/COR:")
print(f"   - Resultado do Chi-quadrado: p-value = {p_raca:.6f}")
print(f"   - Maior e menor taxa de seleção:")
print(f"     • Maior: {raca_taxa['Sim'].idxmax()} ({raca_taxa['Sim'].max():.1f}%)")
print(f"     • Menor: {raca_taxa['Sim'].idxmin()} ({raca_taxa['Sim'].min():.1f}%)")
print(f"     • Gap: {raca_taxa['Sim'].max() - raca_taxa['Sim'].min():.1f} pontos percentuais")
if p_raca < 0.05:
    print(f"   - Conclusão: Associação SIGNIFICANTE - raça/cor está associada à seleção")
    print(f"             ⚠️ ALERTA: Diferenças significantes na taxa de seleção por raça/cor")
else:
    print(f"   - Conclusão: Associação NÃO SIGNIFICANTE")

In [ ]:
# Análise de orientação sexual
ori_taxa = pd.crosstab(df['OrientacaoSexual'], df['Selecionado'], normalize='index') * 100
print(f"\n5. DIFERENÇAS POR ORIENTAÇÃO SEXUAL:")
print(f"   - Resultado do Chi-quadrado: p-value = {p_ori:.6f}")
print(f"   - Maior e menor taxa de seleção:")
print(f"     • Maior: {ori_taxa['Sim'].idxmax()} ({ori_taxa['Sim'].max():.1f}%)")
print(f"     • Menor: {ori_taxa['Sim'].idxmin()} ({ori_taxa['Sim'].min():.1f}%)")
print(f"     • Gap: {ori_taxa['Sim'].max() - ori_taxa['Sim'].min():.1f} pontos percentuais")
if p_ori < 0.05:
    print(f"   - Conclusão: Associação SIGNIFICANTE - orientação sexual está associada à seleção")
else:
    print(f"   - Conclusão: Associação NÃO SIGNIFICANTE")

# Análise de deficiência
def_taxa = pd.crosstab(df['Tem_Deficiencia'], df['Selecionado'], normalize='index') * 100
taxa_com_def = def_taxa.loc['Sim', 'Sim']
taxa_sem_def = def_taxa.loc['Não', 'Sim']
print(f"\n6. DIFERENÇAS POR STATUS DE DEFICIÊNCIA:")
print(f"   - Taxa de seleção com deficiência: {taxa_com_def:.1f}%")
print(f"   - Taxa de seleção sem deficiência: {taxa_sem_def:.1f}%")
print(f"   - Diferença: {abs(taxa_com_def - taxa_sem_def):.1f} pontos percentuais")
print(f"   - Resultado do Chi-quadrado: p-value = {p_def:.6f}")
if p_def < 0.05:
    print(f"   - Conclusão: Associação SIGNIFICANTE - deficiência está associada à seleção")
else:
    print(f"   - Conclusão: Associação NÃO SIGNIFICANTE")

print("\n" + "=" * 100)
print("CONCLUSÃO GERAL")
print("=" * 100)
print(f"\nBasado nos testes estatísticos realizados:")
significant_vars = []
if p_value_idade < 0.05:
    significant_vars.append("Idade")
if p_sexo < 0.05:
    significant_vars.append("Sexo")
if p_raca < 0.05:
    significant_vars.append("Raça/Cor")
if p_ori < 0.05:
    significant_vars.append("Orientação Sexual")
if p_def < 0.05:
    significant_vars.append("Deficiência")

if significant_vars:
    print(f"\nAs seguintes variáveis têm associação significante com a seleção (p < 0.05):")
    for var in significant_vars:
        print(f"  • {var}")
    print(f"\nIsto sugere que o perfil demográfico DOS candidatos diferencia selecionados de não-selecionados.")
else:
    print(f"\nNenhuma variável teve associação significante com a seleção (p ≥ 0.05).")
    print(f"Isto sugere que não há diferenças demográficas significantes entre os grupos.")

## 8. Sugestões de Análises Adicionais

Com base na análise realizada, as seguintes análises seria possível conduzir com informações adicionais:

In [ ]:
print("=" * 100)
print("SUGESTÕES DE ANÁLISES ADICIONAIS")
print("=" * 100)

analises_adicionais = """
1. DESEMPENHO ACADÊMICO (se disponível)
   - Correlação entre notas de prova e variáveis demográficas
   - Análise se desempenho acadêmico explica diferenças nas taxas de seleção
   - Identificar se há viés nos critérios de avaliação

2. ORIGEM GEOGRÁFICA (se disponível)
   - Comparar taxas de seleção por região/estado de origem
   - Investigar se há disparidades em oportunidades por localização
   - Analisar curva de distribuição de candidatos vs. selecionados por região

3. NÍVEL SOCIOECONÔMICO (se disponível)
   - Avaliar como renda familiar influencia as chances de seleção
   - Analisar interações entre status socioeconômico e raça/cor
   - Verificar se bolsas estão distribuindo equitativamente

4. HISTÓRICO EDUCACIONAL (se disponível)
   - Escola de origem (pública vs. privada)
   - Série em que os candidatos estão
   - Desempenho histórico em disciplinas específicas
   - Participação em atividades extracurriculares

5. INTERSECCIONALIDADE
   - Análise de interações entre múltiplas variáveis
   - Ex: mulheres negras vs. mulheres brancas vs. homens negros, etc.
   - Identificar grupos especialmente desfavorecidos ou favorecidos

6. ANÁLISE TEMPORAL (se dados históricos disponíveis)
   - Tendência nas taxas de seleção ao longo dos anos
   - Mudanças na composição demográfica de candidatos e selecionados
   - Avaliar impacto de possíveis políticas de inclusão

7. ANÁLISE DE REGRESSÃO LOGÍSTICA
   - Construir modelo preditivo da probabilidade de seleção
   - Identificar quais fatores são mais importantes
   - Calcular odds ratios para cada variável
   - Validar modelo com técnicas de cross-validation

8. ANÁLISE DE REDES / NETWORK ANALYSIS (se dados de conexões disponíveis)
   - Investigar se há redes sociais que favorecem seleção
   - Identificar padrões de recomendações ou conexões pregresso

9. QUALITATIVO
   - Entrevistas com alunos selecionados vs. não-selecionados
   - Investigar razões percebidas para seleção/não-seleção
   - Análise de comentários ou feedback (se disponível)

10. ANÁLISE DE EQUIDADE
    - Calcular índices de representatividade
    - Comparar representação com população geral
    - Analisar se processo é realmente meritocrático
    - Propor métricas de diversidade e inclusão
"""

print(analises_adicionais)

print("\n" + "=" * 100)
print("QUESTÕES-CHAVE A INVESTIGAR COM DADOS ADICIONAIS")
print("=" * 100)

questoes = """
1. O processo de seleção é meritocrático ou há vieses estruturais?
2. Candidatos de diferentes grupos demográficos têm iguais oportunidades?
3. Se há diferenças, elas são explicadas por fatores acadêmicos ou por discriminação?
4. Como o processo poderia ser melhorado para maior equidade?
5. Qual é a representatividade relativa de cada grupo demográfico?
6. Há grupos que são sistematicamente excluídos ou sub-representados?
7. O processo está atendendo aos objetivos de inclusão e diversidade?
"""

print(questoes)